# Step 1 — Tag Reassignment Causes
**Roadmap reference:** SQL / Python rule-based tagging on assignment history

## Goal
Classify every assignment transition by cause type so every downstream model
has clean, interpretable labels.

## Cause taxonomy
| Tag | Meaning |
|-----|---------|
| System - Auto Route | Initial FNOL assignment fired by routing engine |
| System - Rule Triggered | Mid-claim rule (amount, loss cause) fires reassignment |
| Manual - Supervisor | Supervisor explicitly reassigned via claim screen |
| Manual - Workload | Adjuster initiated transfer due to capacity |
| User Automated | Adjuster used "Use Automated Assignment" — logged as system but user-initiated |
| Fallback Cascade | Primary adjuster unavailable; cascaded to next eligible |
| Named Account Bypass | Named-account override skipped normal routing |

## Important Guidewire limitation
If a user reassigns using *Use Automated Assignment*, the system logs it identically
to a true system assignment. This notebook flags those as `User Automated` using
a heuristic: system-logged event on a non-initial assignment where the prior event
was also system-logged within <2 hours.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

DATA = Path("data")

df = pd.read_csv(DATA / "cp_assignment_history.csv")
print(f"Loaded {len(df)} assignment events across {df['claim_id'].nunique()} claims")
df.head()

Loaded 1116 assignment events across 200 claims


,claim_id,assignment_num,event_type,cause_tag,timestamp,adjuster_id,adjuster_name,group_name,activity_notes,trigger,same_adjuster_returned,final_group
0,CP-0001,1,Assigned,System - Auto Route,04/01/2024 04:00 AM,ADJ-015,Christopher Adams,Subrogation Recovery Unit,Claim created and assigned to Christopher Adam...,System,0,B
1,CP-0001,2,Reassigned,Fallback Cascade,04/01/2024 08:00 AM,ADJ-009,Kevin Walsh,CAT Response Team - Central,Assigned to Kevin Walsh in CAT Response Team -...,Manual,0,B
2,CP-0001,3,Reassigned,Manual - Workload,04/08/2024 12:00 PM,ADJ-004,Marcus Thompson,Columbus Property Team 3,Workload rebalance. Claim moved to Marcus Thom...,System,0,B
3,CP-0002,1,Assigned,System - Auto Route,10/07/2024 12:00 AM,ADJ-001,Sarah Mitchell,West Zone Ops Ohio Team 6,Claim created and assigned to Sarah Mitchell i...,System,0,0
4,CP-0003,1,Assigned,System - Auto Route,03/30/2024 12:00 AM,ADJ-009,Kevin Walsh,CAT Response Team - Central,FNOL intake complete. Assignment routed to Kev...,System,0,B


## Cause tag distribution (as generated)

In [2]:
counts = df["cause_tag"].value_counts().reset_index()
counts.columns = ["Cause Tag", "Count"]
counts["Pct"] = (counts["Count"] / len(df) * 100).round(1)
print(counts.to_string(index=False))

fig = px.bar(counts, x="Cause Tag", y="Count", color="Cause Tag",
             title="Assignment Events by Cause Tag — Commercial Property 2024",
             text="Pct")
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

              Cause Tag  Count  Pct
    Manual - Supervisor    420 37.6
    System - Auto Route    200 17.9
System - Rule Triggered    165 14.8
      Manual - Workload    105  9.4
         User Automated    105  9.4
   Named Account Bypass     71  6.4
       Fallback Cascade     50  4.5


## Flag: User Automated heuristic
System-logged non-initial events within 2 hours of previous system event are re-tagged.

In [3]:
df["timestamp_dt"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %I:%M %p")
df = df.sort_values(["claim_id", "assignment_num"]).reset_index(drop=True)

# Compute hours since previous event
df["prev_trigger"]    = df.groupby("claim_id")["trigger"].shift(1)
df["prev_ts"]         = df.groupby("claim_id")["timestamp_dt"].shift(1)
df["hrs_since_prev"]  = (df["timestamp_dt"] - df["prev_ts"]).dt.total_seconds() / 3600

# Heuristic: non-initial, system-logged, within 2h of previous system event → User Automated
mask = (
    (df["assignment_num"] > 1) &
    (df["trigger"] == "System") &
    (df["prev_trigger"] == "System") &
    (df["hrs_since_prev"] < 2)
)
df.loc[mask, "cause_tag_refined"] = "User Automated"
df["cause_tag_refined"] = df["cause_tag_refined"].fillna(df["cause_tag"])

changed = mask.sum()
print(f"Re-tagged {changed} events as 'User Automated' via heuristic")
print(df["cause_tag_refined"].value_counts().to_string())

Re-tagged 0 events as 'User Automated' via heuristic
cause_tag_refined
Manual - Supervisor        420
System - Auto Route        200
System - Rule Triggered    165
Manual - Workload          105
User Automated             105
Named Account Bypass        71
Fallback Cascade            50


## Breakdown by final group (reassignment severity)

In [4]:
ct = (df.groupby(["final_group", "cause_tag_refined"])
        .size()
        .reset_index(name="count"))
fig = px.bar(ct, x="final_group", y="count", color="cause_tag_refined",
             barmode="stack",
             title="Cause Tag Mix by Reassignment Group",
             labels={"final_group": "Reassignment Group", "count": "Events",
                     "cause_tag_refined": "Cause"},
             category_orders={"final_group": ["0", "A", "B", "C"]})
fig.show()

## Save tagged output

In [5]:
out = df.drop(columns=["timestamp_dt", "prev_trigger", "prev_ts", "hrs_since_prev"])
out.to_csv(DATA / "step1_tagged_assignments.csv", index=False)
print(f"Saved {len(out)} rows → data/step1_tagged_assignments.csv")
out.head()

Saved 1116 rows → data/step1_tagged_assignments.csv


,claim_id,assignment_num,event_type,cause_tag,timestamp,adjuster_id,adjuster_name,group_name,activity_notes,trigger,same_adjuster_returned,final_group,cause_tag_refined
0,CP-0001,1,Assigned,System - Auto Route,04/01/2024 04:00 AM,ADJ-015,Christopher Adams,Subrogation Recovery Unit,Claim created and assigned to Christopher Adam...,System,0,B,System - Auto Route
1,CP-0001,2,Reassigned,Fallback Cascade,04/01/2024 08:00 AM,ADJ-009,Kevin Walsh,CAT Response Team - Central,Assigned to Kevin Walsh in CAT Response Team -...,Manual,0,B,Fallback Cascade
2,CP-0001,3,Reassigned,Manual - Workload,04/08/2024 12:00 PM,ADJ-004,Marcus Thompson,Columbus Property Team 3,Workload rebalance. Claim moved to Marcus Thom...,System,0,B,Manual - Workload
3,CP-0002,1,Assigned,System - Auto Route,10/07/2024 12:00 AM,ADJ-001,Sarah Mitchell,West Zone Ops Ohio Team 6,Claim created and assigned to Sarah Mitchell i...,System,0,0,System - Auto Route
4,CP-0003,1,Assigned,System - Auto Route,03/30/2024 12:00 AM,ADJ-009,Kevin Walsh,CAT Response Team - Central,FNOL intake complete. Assignment routed to Kev...,System,0,B,System - Auto Route
